# (Appendix) Comparing Function Calling Reliability by Model and Provider — NVIDIA build vs OpenRouter

> This notebook is supplementary material for **Section 7** of `M02_3_mcp_a2a_eng.ipynb`.
> Using **repeated experiments**, it **quantitatively compares** how *"whether tool calling works"* is **not determined by model size alone**,
> but also depends on **model characteristics** and **provider (service) availability**.

## Models compared
| # | Provider · Model | Characteristics |
|---|---|---|
| ① | **NVIDIA build** · `deepseek-ai/deepseek-v4.1-flash` ★default | Fixed model, fast responses, parallel tool calls |
| ② | **OpenRouter** · `liquid/lfm-2.5-2.6b:free` (small, 2.6B) | Check whether even a small model makes tool calls |
| ③ | **OpenRouter** · `nvidia/nemotron-3-super-120b-a12b:free` (large) | Check whether a large model calls both tools at once |
| ④ | **OpenRouter** · `openrouter/free` (automatic routing across free models) | The responding model may change on every call |

## Key points — reading along three axes
- **Model-size axis** → ② vs ③: *bigger is not always better*. Each model learns tool calling differently,
  so a small model may call both tools while a large model calls only one.
- **Provider-availability axis** → ②·③ vs ④: pinning a specific `:free` model can mix in `ERROR`s due to overload (503) or
  request limits (429). The router (`openrouter/free`) falls over to an available model, giving high availability but lower reproducibility.
- **Multi (parallel) call axis** → §8: measures whether the model calls several independent tools simultaneously in one response.

> **Runtime environment**: `NVIDIA_API_KEY` and `OPENROUTER_API_KEY` are required in `notebooks/.env` (no local server needed).
> Free-model results **vary with time of day and load.** Check the numbers in the tables below yourself on every run.
> For details, see `env_guides/M02_2_function_calling_eng.md`.

## 0. Environment setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(''))
import utils
utils.reload_env()

from agentic_lib import bootstrap
from agentic_lib.bootstrap import to_text          # provider-agnostic response normalization (strips <think>)

# Packages for comparing NVIDIA build (ChatNVIDIA) + OpenRouter (ChatOpenAI, OpenAI-compatible)
utils.uv_install(['langchain', 'langchain-openai', 'langchain-nvidia-ai-endpoints', 'openai'])
print('Ready')

## 1. Comparison models & model factory

NVIDIA build is accessed via `ChatNVIDIA`, and since OpenRouter is OpenAI-compatible, via `ChatOpenAI(base_url=...)`.
Switching models within OpenRouter only requires changing the one-line `model` string.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Models compared: (provider, model) — all available for free
MODELS = {
    "nvidia":  {"label": "NVIDIA deepseek-v4.1-flash",  "provider": "nvidia",     "model": utils.NVIDIA_MODEL},
    "small":   {"label": "OR lfm-2.5-2.6b (small)",     "provider": "openrouter", "model": "liquid/lfm-2.5-2.6b:free"},
    "large":   {"label": "OR nemotron-3-super (large)", "provider": "openrouter", "model": "nvidia/nemotron-3-super-120b-a12b:free"},
    "router":  {"label": "OR openrouter/free (router)", "provider": "openrouter", "model": "openrouter/free"},
}

def make_chat(provider, model, temperature=0.7):
    """Create a LangChain chat model for the given provider and model (before binding tools).

    timeout: free models can respond slowly because of queues, so allow a generous 2 minutes.
    max_retries: OpenRouter free models easily hit the per-minute request limit (429), so increase automatic retries (exponential backoff).
    """
    if provider == "nvidia":
        # Thinking mode is off by default, same as utils.get_llm('nvidia') (NVIDIA_THINKING=false)
        extra = {} if utils.NVIDIA_THINKING else {"model_kwargs": {"chat_template_kwargs": {"thinking": False}}}
        return ChatNVIDIA(model=model, api_key=utils.NVIDIA_API_KEY,
                          temperature=temperature, max_completion_tokens=1024, **extra)
    return ChatOpenAI(model=model, base_url=utils.OPENROUTER_BASE_URL, api_key=utils.OPENROUTER_API_KEY,
                      temperature=temperature, max_tokens=1024, timeout=120, max_retries=5)

def make_llm(key, temperature=0.7):
    """Create a tool-bound LLM from a MODELS key.
    (TOOLS is defined in the next cell and is referenced at call time.)
    """
    m = MODELS[key]
    return make_chat(m["provider"], m["model"], temperature).bind_tools(TOOLS)

for k, m in MODELS.items():
    print(f"  {k:7s} {m['provider']:11s} {m['model']}")

## 2. Tool definitions + classifier (native `tool_calls` first, `<tool_call>` parsing fallback)

We prepare a request that **requires** tool calls, and classify each response into one of the following four categories.

| Category | Meaning |
|------|------|
| `TOOL_OK` | Obtained an **executable tool call** — native `tool_calls` or parsed `<tool_call>` JSON ✅ |
| `MALFORMED` | Tool-call text is present, but the JSON is **broken** and cannot be parsed ⚠️ |
| `NO_TOOL` | Did not attempt a tool and just answered in prose |
| `ERROR` | Exception (overload 503, request limit 429, timeout, etc.) |

`extract_calls()` extracts executable calls from the response. Cloud APIs usually respond with **native `tool_calls`
(source=native)**, but if a model leaks its tool calls into the body text, it falls back to regex parsing of `<tool_call>`
(source=parsed). `temperature=0.7` is used to observe variation across repetitions.

In [ ]:
import math, time, json as _json, re
from collections import Counter
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

@tool
def get_weather(city: str) -> str:
    """Look up the current weather for a city."""
    return f"Weather in {city}: sunny, 22°C"

# Expose only math symbols that are safe for eval (in case the model uses sqrt/pow etc. directly)
_MATH = {k: getattr(math, k) for k in
         ['sqrt','pow','sin','cos','tan','log','log10','exp','pi','e','factorial','floor','ceil','fabs']}

@tool
def calculator(expression: str) -> str:
    """Perform a math calculation. (e.g., 'sqrt(144)', '12 * 12')"""
    try:
        return f"{expression} = {eval(expression, {'__builtins__': {}}, _MATH)}"
    except Exception as e:
        return f"Error: {e}"

TOOLS = [get_weather, calculator]
TOOL_FUNCS = {t.name: t for t in TOOLS}
# This request needs both tools: 'weather lookup' + 'square root calculation' (for measuring completeness).
REQUIRED_TOOLS = {"get_weather", "calculator"}
PROMPT = [SystemMessage(content="You must answer using the provided tools."),
          HumanMessage(content="Tell me the weather in Seoul and calculate the square root of 144.")]

_TOOL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)

def parse_text_calls(text):
    """Extract and validate <tool_call> JSON blocks from body text and return them as a list of (name, args)."""
    calls = []
    for m in _TOOL_RE.finditer(text):
        try:
            obj = _json.loads(m.group(1))
        except Exception:
            continue                                  # broken JSON → not executable
        if isinstance(obj, dict) and "name" in obj:
            calls.append((obj["name"], obj.get("arguments", obj.get("parameters", {})) or {}))
    return calls

def extract_calls(resp):
    """Extract executable tool calls. Returns: (calls, source) — source: native|parsed|None"""
    native = getattr(resp, "tool_calls", None) or []
    if native:
        return [(t["name"], t.get("args", {})) for t in native], "native"
    parsed = parse_text_calls(to_text(resp.content))  # fallback for tool calls leaked into the body
    if parsed:
        return parsed, "parsed"
    return [], None

def classify(resp):
    """Classify a response as TOOL_OK / MALFORMED / NO_TOOL."""
    calls, source = extract_calls(resp)
    if calls:
        return "TOOL_OK", source, [n for n, _ in calls]
    text = to_text(resp.content)
    if ("<tool_call>" in text) or ('"arguments"' in text) or ('"parameters"' in text):
        return "MALFORMED", None, None               # intended a call but parsing failed
    return "NO_TOOL", None, None

def run_experiment(llm_with_tools, n=10, pause=0.0):
    """Repeat the same request n times and return the classification results (plus called tool names and responding model).

    pause: wait time between calls (seconds). Used to avoid the per-minute request limit (429) of OpenRouter free models.
    """
    results = []
    for i in range(n):
        if i and pause:
            time.sleep(pause)
        try:
            resp = llm_with_tools.invoke(PROMPT)
            kind, source, names = classify(resp)
            served = resp.response_metadata.get("model_name", "")   # the model that actually responded (to check the router)
            detail = f"{names} ({source})" if names else to_text(resp.content)[:60].replace(chr(10), ' ')
            detail += f"  · {served}" if served else ""
        except Exception as e:
            kind, source, names, detail = "ERROR", None, None, repr(e)[:80]
        results.append({"kind": kind, "names": names or [], "detail": detail})
        print(f"  run {i+1:2d}: {kind:10s} | {detail}")
    return results

def summarize(results, label):
    """Aggregate the valid tool_calls rate + the 'called both tools' (completeness) rate."""
    c = Counter(r["kind"] for r in results)
    n = len(results)
    ok = c.get("TOOL_OK", 0)
    both = sum(1 for r in results if REQUIRED_TOOLS.issubset(set(r["names"])))  # completeness metric
    print(f"\n[{label}] valid tool_calls {ok}/{n} = {ok/n*100:.0f}%  ·  "
          f"both tools called {both}/{n} = {both/n*100:.0f}%  ·  categories {dict(c)}")
    return {"label": label, "n": n, "ok": ok, "both": both, "counts": dict(c)}

print("Tools/helpers ready:", [t.name for t in TOOLS])

## 3. Experiment ① — NVIDIA build · `deepseek-v4.1-flash` (default) ★

This is the default model for this course. Because the model is fixed, results are highly reproducible, and it often calls
both tools **in parallel** in a single response.

In [ ]:
N = 10
SUMMARY = {}   # key → summarize() result
OR_PAUSE = 3.0 # interval between OpenRouter calls (seconds): stay under the free-model per-minute limit (~20/min)

def run_model(key, n=N):
    """Run the repeated experiment with MODELS[key] and store the summary in SUMMARY."""
    m = MODELS[key]
    print(f"=== {m['label']} repeated experiment (N={n}) — {m['provider']} · {m['model']} ===")
    pause = OR_PAUSE if m["provider"] == "openrouter" else 0.0
    res = run_experiment(make_llm(key), n, pause)
    SUMMARY[key] = summarize(res, m["label"])
    return res

res_nvidia = run_model("nvidia")

## 4. Experiment ② — OpenRouter · small `lfm-2.5-2.6b` (model-size axis)

This is a **small model** with 2.6B parameters. Even a small model can produce spec-compliant `tool_calls` if it has learned
tool calling well. Since it is a pinned free model, `ERROR`s (429) may be mixed in at busy times.

In [ ]:
res_small = run_model("small")

## 5. Experiment ③ — OpenRouter · large `nemotron-3-super-120b` (model-size axis)

This is a **large model** with 120B parameters. It produces valid tool calls, but we use the **"both tools called (completeness)"** rate
to check whether it tends to call **only one tool per response** for a request that needs two tools.
A model that calls only one will eventually handle both tasks if the agent loop runs several times, but the number of round trips increases.

In [ ]:
res_large = run_model("large")

incomplete = [x for x in res_large if x["kind"] == "TOOL_OK" and not REQUIRED_TOOLS.issubset(set(x["names"]))]
if incomplete:
    print(f"\n--- Examples of responses that called only some of the two tools (incomplete): {len(incomplete)}/{N} ---")
    for r in incomplete[:3]:
        print("  •", r["detail"][:120])

## 6. Experiment ④ — OpenRouter · `openrouter/free` router (provider-availability axis)

The **Free Models Router** picks an available free model for each request. Looking at the `· model name` at the end of each run,
you can see that the responding model changes from call to call. Even if a particular model is overloaded, it falls over to another model,
so there are few `ERROR`s — but because the model changes, **reproducibility is low.** The router cannot avoid the account-level
per-minute request limit (429) either, so this notebook waits `OR_PAUSE` (3 seconds) between OpenRouter calls.

In [ ]:
res_router = run_model("router")

served = Counter(r["detail"].split("· ")[-1] for r in res_router if "· " in r["detail"])
print("\nResponding model distribution:", dict(served))

### 6-1. Real tool execution loop (NVIDIA `deepseek-v4.1-flash`)

This shows that the extracted calls are **actually executable**: parse the response → run the Python tools → feed the results back to the model to generate the final answer.

In [ ]:
llm_nvidia = make_llm("nvidia")
resp = llm_nvidia.invoke(PROMPT)
calls, source = extract_calls(resp)
print(f"1) Extracted tool calls (source={source}):")
for name, args in calls:
    print(f"   - {name}({args})")

print("\n2) Actually running the tools:")
tool_results = []
for name, args in calls:
    out = TOOL_FUNCS[name].invoke(args)
    tool_results.append(f"{name} -> {out}")
    print(f"   - {out}")

print("\n3) Feeding the results back to the model to generate the final answer:")
followup = PROMPT + [
    HumanMessage(content="Here are the tool execution results:\n" + "\n".join(tool_results) +
                 "\nUsing only these results, give the user a natural final answer in English.")]
# The final answer is text-only, without tools (default nvidia/deepseek-v4.1-flash)
final = utils.get_llm("nvidia", temperature=0.3).invoke(followup)
print("   ", to_text(final.content).strip()[:300])

## 7. Comparison results (4-way)

In [ ]:
def bar(pct, width=20):
    fill = int(round(pct / 100 * width))
    return "█" * fill + "·" * (width - fill)

print(f"{'Provider · Model':<30}{'Valid tool_calls':<26}{'Both tools (completeness)':<26}{'ERROR'}")
print("-" * 92)
for key in ["nvidia", "small", "large", "router"]:
    s = SUMMARY.get(key)
    if not s or s.get("n", 0) == 0:
        continue
    ok_pct = s["ok"] / s["n"] * 100
    both_pct = s["both"] / s["n"] * 100
    err = s["counts"].get("ERROR", 0)
    print(f"{s['label']:<30}{bar(ok_pct)} {ok_pct:3.0f}%   {bar(both_pct)} {both_pct:3.0f}%   {err}/{s['n']}")

print("\nInterpretation — the axes of function calling reliability:")
print("  (model-size axis)  small lfm-2.5-2.6b  vs  large nemotron-3-super  ← same provider (OpenRouter), different models")
print("     · 'bigger is always better' does not hold. Completeness (calling both tools) depends on how tool calling was learned")
print("  (provider-availability axis) pinned :free model  vs  openrouter/free router")
print("     · pinned free models can mix in ERRORs from overload (503) / limits (429); the router offers high availability but the model changes")
print("  ⇒ Don't choose a model for your agent by name or size alone — repeat the same request N times and measure it yourself.")

## 8. Multi (Parallel) Function Calling — a key capability for agents

The earlier experiments (§3–7) looked at *"does it produce valid tool calls"* (reliability). Here we go one step further
and ask **"can it call multiple tools simultaneously in a single response?"**

- **Single function calling**: requests **only one** tool per response → the rest in later turns.
- **Multi (parallel) function calling**: requests several **independent** tools **simultaneously in one response**.

### Why it matters for agents
Independent subtasks (e.g., "weather lookup" + "square root calculation") have no ordering dependency. Multi-calling bundles them
into **a single turn**, which gives:
- **Fewer round trips and lower latency** — fewer LLM calls make it faster and cheaper.
- **Concurrent planning** — the model plans all the needed actions at once, simplifying tool orchestration.
- **ReAct/tool-agent efficiency** — it collects several observations at once and moves on to the next reasoning step.

A model that supports only single calls will eventually handle both tasks if the agent loop runs several times, but the extra turns make it
slower and add more points of failure. That is why **parallel calling capability is an important factor in practical agent design**.

> 📌 Whether a model does multi or single calls depends jointly on *model, prompt, and temperature*, and there is **nondeterministic variation**. Therefore,
> before using a particular model in an agent, it is important to make a habit of **measuring it yourself N times as below** to confirm that parallel calling is stable.

In [ ]:
# === §8. Measuring multi (parallel) function calling — how many tools are called in one response ===
MULTI_USER = "Tell me the weather in Seoul and calculate the square root of 144."
P_MULTI = [SystemMessage(content="You must use the provided tools."), HumanMessage(content=MULTI_USER)]

def count_parallel(llm_with_tools, prompt, n=8, pause=0.0):
    """Send the same request n times and record 'the number of tool_calls returned in a single response'."""
    counts = []
    for i in range(n):
        if i and pause:
            time.sleep(pause)   # avoid the OpenRouter per-minute limit (429)
        try:
            resp = llm_with_tools.invoke(prompt)
            counts.append(len(getattr(resp, "tool_calls", None) or []))
        except Exception as e:
            counts.append(f"ERR:{type(e).__name__}")
    return counts

def verdict(counts):
    """Judge by the number of calls per response: any 2+ means parallel-capable, all 1 means single; variation is shown separately."""
    nums = [c for c in counts if isinstance(c, int)]
    if not nums:
        return "Not measurable (all errors) ❓"
    if all(c == 0 for c in nums):
        return "No tool calls ❓"
    multi = sum(1 for c in nums if c >= 2)
    if multi == 0:
        return "Single only ⚠️"
    if multi == len(nums):
        return "Multi (parallel) ✅"
    if multi * 2 >= len(nums):
        return f"Mostly multi (with variation: {multi}/{len(nums)}) ✅"
    return f"Partly multi (mostly single: parallel {multi}/{len(nums)}) ⚠️"

Nm = 8
print(f"Request: {MULTI_USER}\n(requires both independent tools get_weather and calculator; {Nm} repetitions each, temperature=0)\n")
PARALLEL = {}
for key, m in MODELS.items():
    counts = count_parallel(make_llm(key, temperature=0), P_MULTI, Nm,
                            OR_PAUSE if m["provider"] == "openrouter" else 0.0)
    PARALLEL[key] = verdict(counts)
    print(f"[{m['label']:30s}] tool_calls per response {counts} → {PARALLEL[key]}")

print("\nInterpretation:")
print("  · Even for the same request, depending on the model, both tools are called in parallel in one response (multi) or one at a time (single)")
print("  · The number of calls can vary from run to run depending on model, prompt, and temperature (nondeterministic variation)")
print("  ⇒ Don't conclude 'this model can/can't'; measure the model you will use yourself. Parallel calls cut round trips and latency, which matters for agents.")

## 9. Summary

Function calling reliability depends on **several axes** together.

1. **Model-size axis** — Experiment ② vs ③ (same OpenRouter, different models):
   - A larger model does not automatically mean higher completeness. Depending on how tool calling was learned,
     a small model may call both tools while a large model calls only one.
2. **Provider-availability axis** — Experiments ②·③ vs ④:
   - Pinning a specific `:free` model can mix in **`ERROR`**s due to overload (503) or request limits (429).
   - The `openrouter/free` router falls over to an available model, so **availability** is high, but the model changes on every call, so **reproducibility** is low.
   - **Pinning the model**, as with NVIDIA build, gives high reproducibility, making it suitable as the course's default model.
3. **Multi (parallel) call axis** — §8 (how many tools are called in one response):
   - Parallel calls reduce round trips and latency and enable concurrent planning, making them **important in practical agent design**.
   - The number of calls has **nondeterministic variation**, so it is a good habit to **measure repeatedly yourself** with the model you will use.

**Conclusion**: Choose the model for your agent **by measurement, not by name or size**. This course uses
**NVIDIA build `deepseek-ai/deepseek-v4.1-flash`** as the default, since its tool calling is stable, it supports parallel calls, and its results are highly reproducible,
and the main notebook (`M02_3_mcp_a2a_eng.ipynb` §7) uses this default provider as is for its agent via `utils.get_llm()`.